# Research Agent Review and Selection

このノートブックは、Researcher Agentが生成した改善提案を確認し、実装する提案を選択するためのインターフェースです。

## 使用方法

1. 最新の研究計画ファイルを読み込み
2. 各提案の詳細を確認
3. 実装したい提案を選択
4. 選択した提案を保存

In [ ]:
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from datetime import date

## 1. 最新の研究計画を読み込み

In [ ]:
# 研究計画ファイルのパスを設定
research_dir = Path("../data/output/research_reports")
today = date.today().isoformat()

# 最新の研究計画ファイルを探す
plan_files = list(research_dir.glob(f"**/research_plan_*.yaml"))
if plan_files:
    latest_plan_file = max(plan_files, key=lambda p: p.stat().st_mtime)
    print(f"最新の研究計画: {latest_plan_file}")
else:
    latest_plan_file = research_dir / f"research_plan_{today}.yaml"
    print(f"デフォルトパス: {latest_plan_file}")

# 研究計画を読み込み
if latest_plan_file.exists():
    with open(latest_plan_file, 'r', encoding='utf-8') as f:
        research_plan = yaml.safe_load(f)
    
    print(f"実験ID: {research_plan.get('experiment_id')}")
    print(f"提案数: {len(research_plan.get('proposals', []))}")
else:
    print("⚠️ 研究計画ファイルが見つかりません")
    research_plan = None

## 2. 提案の概要

In [ ]:
if research_plan and 'proposals' in research_plan:
    proposals = research_plan['proposals']
    
    # 提案の概要をDataFrameで表示
    summary_data = []
    for proposal in proposals:
        # 期待効果を文字列に変換
        effects = ', '.join([f"{k}: {v}" for k, v in proposal.get('expected_effect', {}).items()])
        
        summary_data.append({
            'ID': proposal.get('id', ''),
            'タイトル': proposal.get('title', ''),
            '期待効果': effects,
            '実装工数': proposal.get('human_effort', ''),
            'リスク': proposal.get('risk', '')[:50] + '...' if len(proposal.get('risk', '')) > 50 else proposal.get('risk', '')
        })
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
else:
    print("提案データがありません")

## 3. 詳細な提案内容

In [ ]:
if research_plan and 'proposals' in research_plan:
    for i, proposal in enumerate(proposals):
        proposal_id = proposal.get('id', f'Proposal {i+1}')
        
        # Markdown形式で提案詳細を表示
        markdown_content = f"""
### 🔹 提案 {proposal_id}: {proposal.get('title', '')}

**根拠**: {proposal.get('rationale', '')}

**変更内容**:
"""
        
        for j, change in enumerate(proposal.get('changes', []), 1):
            action = change.get('action', '')
            params = change.get('params', {})
            
            markdown_content += f"""
{j}. **{action}**
"""
            if params:
                for param_name, param_value in params.items():
                    markdown_content += f"   - `{param_name}`: {param_value}\n"
            
            if 'description' in change:
                markdown_content += f"   - {change['description']}\n"
        
        markdown_content += f"""
**期待効果**:
"""
        for metric, effect in proposal.get('expected_effect', {}).items():
            markdown_content += f"- **{metric}**: {effect}\n"
        
        markdown_content += f"""
**リスク**: {proposal.get('risk', '')}

**実装工数**: {proposal.get('human_effort', '')}

---
"""
        
        display(Markdown(markdown_content))

## 4. 提案選択インターフェース

In [ ]:
if research_plan and 'proposals' in research_plan:
    # 提案選択用のラジオボタン
    proposal_options = [(f"{p.get('id', i)}: {p.get('title', f'Proposal {i+1}')}", i) 
                       for i, p in enumerate(proposals)]
    proposal_options.append(("実装しない (スキップ)", -1))
    
    proposal_selector = widgets.RadioButtons(
        options=proposal_options,
        description='選択:',
        disabled=False
    )
    
    display(proposal_selector)
    
    # 選択した提案の詳細表示
    def show_selected_proposal(change):
        selected_idx = proposal_selector.value
        if selected_idx >= 0 and selected_idx < len(proposals):
            selected_proposal = proposals[selected_idx]
            print(f"\n選択された提案: {selected_proposal.get('id')} - {selected_proposal.get('title')}")
            print(f"根拠: {selected_proposal.get('rationale')}")
        elif selected_idx == -1:
            print("\n現在の提案をスキップします")
    
    proposal_selector.observe(show_selected_proposal, names='value')
else:
    proposal_selector = None
    print("選択可能な提案がありません")

## 5. 選択した提案を保存

In [ ]:
def save_selected_proposal():
    """選択した提案を保存する"""
    if not research_plan or not proposal_selector:
        print("❌ 保存可能な提案がありません")
        return
    
    selected_idx = proposal_selector.value
    
    if selected_idx == -1:
        print("✅ 提案をスキップしました")
        return
    
    if selected_idx < 0 or selected_idx >= len(proposals):
        print("❌ 無効な選択です")
        return
    
    selected_proposal = proposals[selected_idx]
    
    # 選択した提案を保存
    selected_plan = {
        'experiment_id': research_plan.get('experiment_id'),
        'base_commit': research_plan.get('base_commit'),
        'selected_proposal': selected_proposal,
        'selection_timestamp': pd.Timestamp.now().isoformat(),
        'selected_by': 'human_review'
    }
    
    # 保存先パス
    actions_dir = Path("../actions")
    actions_dir.mkdir(exist_ok=True)
    
    output_file = actions_dir / f"selected_plan_{date.today().isoformat()}.yaml"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        yaml.dump(selected_plan, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✅ 選択した提案を保存しました: {output_file}")
    print(f"提案: {selected_proposal.get('id')} - {selected_proposal.get('title')}")
    
    # 実装のための次のステップを表示
    print("\n📋 次のステップ:")
    print("1. パラメータファイルを更新")
    print("2. 必要に応じてコードを修正")
    print("3. harmonizer を再実行")
    print("4. 結果を評価")

# 保存ボタン
save_button = widgets.Button(
    description='選択した提案を保存',
    button_style='success',
    icon='check'
)

save_button.on_click(lambda b: save_selected_proposal())

display(save_button)

## 6. 提案の手動編集（オプション）

In [ ]:
# 選択した提案のパラメータを手動で調整したい場合
if research_plan and proposal_selector and proposal_selector.value >= 0:
    selected_idx = proposal_selector.value
    if selected_idx < len(proposals):
        selected_proposal = proposals[selected_idx]
        
        print("選択した提案の編集:")
        print("\n現在のパラメータ:")
        
        for change in selected_proposal.get('changes', []):
            if 'params' in change:
                for param_name, param_value in change['params'].items():
                    print(f"  {param_name}: {param_value}")
        
        print("\n💡 パラメータを調整したい場合は、YAMLファイルを直接編集してください")
        print(f"   ファイル: {latest_plan_file}")

## 7. 履歴とトラッキング

In [ ]:
# 過去の選択履歴を表示
actions_dir = Path("../actions")
if actions_dir.exists():
    selected_files = list(actions_dir.glob("selected_plan_*.yaml"))
    
    if selected_files:
        print("📊 過去の選択履歴:")
        
        for file_path in sorted(selected_files, reverse=True)[:5]:  # 最新5件
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    past_selection = yaml.safe_load(f)
                
                proposal = past_selection.get('selected_proposal', {})
                timestamp = past_selection.get('selection_timestamp', 'Unknown')
                
                print(f"  {file_path.stem}: {proposal.get('title', 'Unknown')} ({timestamp[:10]})")
            except Exception as e:
                print(f"  {file_path.stem}: 読み込みエラー ({e})")
    else:
        print("📝 選択履歴がありません")
else:
    print("📝 actionsディレクトリが存在しません")

---

## 使用方法まとめ

1. **提案確認**: セクション2で概要を、セクション3で詳細を確認
2. **提案選択**: セクション4のラジオボタンで実装したい提案を選択
3. **保存**: セクション5の保存ボタンで選択を確定
4. **実装**: 生成されたYAMLファイルに基づいてパラメータやコードを更新

提案の実装後は、harmonizerを再実行して効果を確認してください。